# 04 - Multi-Region MMM Comparison

Fit separate MMM models per region and compare results.

**Note:** True hierarchical MMM with `dims` requires pymc-marketing >= 0.8.0

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az

from pymc_marketing.mmm import (
    MMM,
    GeometricAdstock,
    LogisticSaturation,
)

SEED = 42
rng = np.random.default_rng(SEED)
az.style.use("arviz-darkgrid")
print("OK")

## 2. Load Data

In [ ]:
df = pd.read_parquet("../data/processed/mmm_data.parquet")
print(f"Shape: {df.shape}")

df["date"] = pd.to_datetime(df["DATE_DAY"])

# Select top N regions
N_REGIONS = 3
region_revenue = df.groupby("TERRITORY_NAME")["ALL_PURCHASES_ORIGINAL_PRICE"].sum()
top_regions = region_revenue.nlargest(N_REGIONS).index.tolist()

print(f"Top regions: {top_regions}")

In [ ]:
SPEND_COLS = [
    "GOOGLE_PAID_SEARCH_SPEND", "GOOGLE_SHOPPING_SPEND",
    "META_FACEBOOK_SPEND", "META_INSTAGRAM_SPEND", "TIKTOK_SPEND"
]
TARGET_COL = "ALL_PURCHASES_ORIGINAL_PRICE"

def prepare_region_data(df, region):
    """Prepare weekly data for a single region."""
    df_r = df[df["TERRITORY_NAME"] == region].copy()
    df_r["week"] = df_r["date"].dt.to_period("W").dt.start_time
    
    agg_cols = [c for c in SPEND_COLS + [TARGET_COL] if c in df_r.columns]
    df_weekly = df_r.groupby("week")[agg_cols].sum().reset_index()
    df_weekly = df_weekly.sort_values("week").reset_index(drop=True)
    
    return df_weekly

# Prepare data per region
region_data = {r: prepare_region_data(df, r) for r in top_regions}
for r, d in region_data.items():
    print(f"{r}: {len(d)} weeks")

## 3. Fit Models Per Region

In [ ]:
%%time
models = {}
results = []

for region in top_regions:
    print(f"\n{'='*50}")
    print(f"Fitting: {region}")
    print(f"{'='*50}")
    
    df_weekly = region_data[region]
    
    # Prepare X and y
    channels = [c for c in SPEND_COLS if c in df_weekly.columns]
    X = df_weekly[["week"] + channels].copy().fillna(0)
    y = df_weekly[TARGET_COL].values
    
    # Create model
    mmm = MMM(
        date_column="week",
        channel_columns=channels,
        adstock=GeometricAdstock(l_max=8),
        saturation=LogisticSaturation(),
        yearly_seasonality=2,
    )
    
    # Fit
    mmm.fit(
        X=X,
        y=y,
        chains=2,
        draws=500,
        target_accept=0.9,
        random_seed=rng,
    )
    
    # Store
    models[region] = mmm
    
    # Posterior predictive
    mmm.sample_posterior_predictive(X, extend_idata=True)
    y_pred = mmm.idata.posterior_predictive["y"].mean(dim=["chain", "draw"]).values
    
    r2 = 1 - ((y - y_pred)**2).sum() / ((y - y.mean())**2).sum()
    results.append({"region": region, "r2": r2, "n_weeks": len(y)})
    
    print(f"R²: {r2:.3f}")

print("\nAll models fitted.")

## 4. Compare Results

In [ ]:
results_df = pd.DataFrame(results)
results_df

In [ ]:
# Channel contributions per region
contrib_data = []

for region, mmm in models.items():
    try:
        contributions = mmm.compute_channel_contribution_original_scale()
        mean_contrib = contributions.mean(dim=["chain", "draw"])
        total = mean_contrib.sum(dim="date")
        
        for ch in total.channel.values:
            contrib_data.append({
                "region": region,
                "channel": ch.replace("_SPEND", ""),
                "contribution": float(total.sel(channel=ch).values)
            })
    except Exception as e:
        print(f"{region}: {e}")

contrib_df = pd.DataFrame(contrib_data)
contrib_df.head(10)

In [ ]:
if len(contrib_df) > 0:
    pivot = contrib_df.pivot(index="channel", columns="region", values="contribution")
    
    fig, ax = plt.subplots(figsize=(12, 6))
    pivot.plot(kind="barh", ax=ax)
    ax.set_xlabel("Total Contribution")
    ax.set_title("Channel Contributions by Region")
    plt.tight_layout()
    plt.show()

In [ ]:
# ROI per region
roi_data = []

for region, mmm in models.items():
    df_weekly = region_data[region]
    channels = [c for c in SPEND_COLS if c in df_weekly.columns]
    
    try:
        contributions = mmm.compute_channel_contribution_original_scale()
        mean_contrib = contributions.mean(dim=["chain", "draw"])
        total = mean_contrib.sum(dim="date")
        
        for ch in channels:
            spend = df_weekly[ch].sum()
            contrib = float(total.sel(channel=ch).values) if ch in total.channel.values else 0
            roi = contrib / spend if spend > 0 else 0
            roi_data.append({
                "region": region,
                "channel": ch.replace("_SPEND", ""),
                "spend": spend,
                "contribution": contrib,
                "roi": roi
            })
    except Exception as e:
        print(f"{region}: {e}")

roi_df = pd.DataFrame(roi_data)
roi_df

In [ ]:
if len(roi_df) > 0:
    pivot_roi = roi_df.pivot(index="channel", columns="region", values="roi")
    
    fig, ax = plt.subplots(figsize=(12, 6))
    pivot_roi.plot(kind="barh", ax=ax)
    ax.axvline(x=1.0, color="gray", linestyle="--", label="Break-even")
    ax.set_xlabel("ROI")
    ax.set_title("Channel ROI by Region")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. Save

In [ ]:
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

# Save each model
for region, mmm in models.items():
    safe_name = region.replace(" ", "_").lower()
    mmm.idata.to_netcdf(MODEL_DIR / f"mmm_{safe_name}_trace.nc")
    print(f"Saved: mmm_{safe_name}_trace.nc")

# Save ROI comparison
roi_df.to_csv(MODEL_DIR / "roi_multiregion.csv", index=False)
print("Saved: roi_multiregion.csv")

## Summary

Separate MMM models fitted per region.

**Key insights:**
- Compare channel effectiveness across regions
- Identify which regions respond better to each channel
- Use ROI differences to inform regional budget allocation